# Evaluate Term Dispersion Scores on the GENIA Corpus Data and Reproduce Results Reported in the Mannuscript 

Description: Evaluate the following term dispersion score/keyword extraction methods on the Genia corpus data:
- Inverse Document Frequency (IDF)
- Inverse Collection Frequency (ICF)
- Chi-square
- Church and Gale (CG)
- Irvine and Callison-Burch (ICB)
- Derivation of Proportions (DoP)
- Residual ICF (RICF)
- KeyBERT
- KeyLLM

Calculate average P@k scores for each scoring function using the GENIA terms as ground truth. Also, evaluate scoring functions for their ability to filter out stopwords.

This version of the code excludes singletons from the analysis.

## Preliminaries

In [1]:
# Imports
import sys
import os
import pickle
import json
import pandas as pd
sys.path.append('../../../')
import wordstats
from sklearn.feature_extraction.text import CountVectorizer
import random
import numpy as np
import scipy
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from io import StringIO
from numpy import nan
from tqdm import tqdm
import rbo

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pasheridan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the GENIA Corpus Data

In particular, we load the preprocessed GENIA corpus documents, and gold standard biological terms (i.e., lexical units) and their associated semantic classes (i.e., sems) and associated high-level class (i.e., amino_acid, nucleotide, multi_cell, cell, and other).

First, load the corpus docs, and the lexical units. Then hardcode the high-level semantic classes.

In [2]:
# Load the preprocessed GENIA corpus documents
genia_corpus_path = '../../1-preprocessing/GENIAcorpus3.02-preprocessed.json'

with open(genia_corpus_path, "r") as j:
  genia_corpus = json.loads(j.read())

# Load gold standard terms 
genia_keywords_path = '../../1-preprocessing/GENIAcorpus3.02-keywords.tsv'

with open(genia_keywords_path, "r") as c:
  genia_lexical_units_and_sems = pd.read_csv(c, sep='\t')

genia_lexical_units = genia_lexical_units_and_sems.lex.to_numpy()

# Hardcode the low-level semantic classes and their associated high-level abstract semantic classes
amino_acid_sems = ['G#amino_acid_monomer', 'G#peptide', 'G#protein_N/A',
              'G#protein_complex', 'G#protein_domain_or_region',
              'G#protein_family_or_group', 'G#protein_molecule',
              'G#protein_substructure', 'G#protein_subunit',
              'G#other_organic_compound', 'G#organic', 'G#inorganic', 'G#atom',
              'G#carbohydrate', 'G#lipid']
nucleotide_sems = ['G#nucleotide', 'G#polynucleotide', 'G#DNA_N/A',
        'G#DNA_domain_or_region', 'G#DNA_family_or_group', 'G#DNA_molecule',
        'G#DNA_substructure', 'G#RNA_N/A', 'G#RNA_domain_or_region',
        'G#RNA_family_or_group', 'G#RNA_molecule', 'G#RNA_substructure']
multi_cell_sems = ['G#virus', 'G#mono_cell', 'G#multi_cell', 'G#body_part', 'G#tissue']
cell_sems = ['G#cell_type', 'G#cell_component', 'G#cell_line', 'G#other_artificial_source']
other_sems = ['G#other_name']
high_level_semantic_class_names = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']
high_level_semantic_class_lex_units = [genia_lexical_units, amino_acid_sems, nucleotide_sems, multi_cell_sems, cell_sems, other_sems]

Process high-level semantic classes.

In [3]:
# Collect lexical units belonging to a given high-level semantic class
def get_high_level_semantic_class_words(high_level_class_lst):
  words = []
  for k, v in lex_sem_dct.items():
    if v in high_level_class_lst:
      words.append(k)
  return words

# Create dictionary of lexical units and their associated semantic classes
sem = np.array(genia_lexical_units_and_sems['sem'])
lex = np.array(genia_lexical_units_and_sems['lex'])
lex_sem_dct = dict(zip(lex, sem))

# Create data frame of lexical units, semantic classes, and high-level semantic classes
lex_size = len(genia_lexical_units_and_sems) # Number of lexical units in the vocabulary
high_level_sems_lst = [] # Initialize list for recording high-level semantic classes

# For each term in the vocab, identify low-level semantic class with high-level one 
for index in range(lex_size):
    low_level_sem = genia_lexical_units_and_sems.iloc[index, 1]
    if low_level_sem in amino_acid_sems:
        high_level_sems_lst.append('amino_acid')
    elif low_level_sem in nucleotide_sems:
        high_level_sems_lst.append('nucleotide')
    elif low_level_sem in multi_cell_sems:
        high_level_sems_lst.append('multi_cell')
    elif low_level_sem in cell_sems:
        high_level_sems_lst.append('cell')
    else:
        high_level_sems_lst.append('other')

# Add high-level semantic classes to data frame
genia_lexical_units_and_sems['class'] = high_level_sems_lst

# Print to console:
display(genia_lexical_units_and_sems)

,lex,sem,class
0,IL-2_gene_expression_lex,G#other_name,other
1,IL-2_gene_lex,G#DNA_domain_or_region,nucleotide
2,NF-kappa_B_activation_lex,G#other_name,other
3,NF-kappa_B_lex,G#protein_molecule,amino_acid
4,CD28_lex,G#protein_molecule,amino_acid
...,...,...,...
31782,gp160-induced_AP-1_complex_lex,G#protein_complex,amino_acid
31783,protein_synthesis-independent_lex,G#other_name,other
31784,calcium_channel_blocker_lex,G#other_organic_compound,amino_acid
31785,anti-CD3-induced_interleukin-2_secretion_lex,G#other_name,other


## Prepare the GENIA Corpus Data for Analysis

Prepare the corpus vocabulary.

In [4]:
# Compile the GENIA corpus vocabulary
pre_vocab = []
for i in range(len(genia_corpus)):
  pre_vocab.append(genia_corpus[i].split())

vocab = []
for i in range(len(pre_vocab)):
  for j in range(len(pre_vocab[i])):
    vocab.append(pre_vocab[i][j])

vocab = list(set(vocab))
vocab.sort()

# Helper function to ensure that CountVectorizer doesn't ignore any terms
def analyzer_custom(doc):
  return doc.split()

# Convert GENIA documents into term-in-document matrix of token counts.
counter = CountVectorizer(lowercase=False, vocabulary=vocab, analyzer=analyzer_custom)
collection = counter.transform(genia_corpus)

## Evaluate Term Dispersion Scores for Selected Measures

Calculate bag-of-words model word statistics and related quantities.

In [5]:
# Calculate word statistics and related quantities
m = len(counter.get_feature_names_out()) # vocab size
d = collection.shape[0] # collection size
N_i = wordstats.get_Ni(collection)
N_j = wordstats.get_Nj(collection)
N = wordstats.get_N(N_j)
B_ij = wordstats.get_Bij(collection)
B_i = wordstats.get_Bi(B_ij)
B_j = wordstats.get_Bj(B_ij)
DF = wordstats.get_DF(B_i, d)
CF = wordstats.get_CF(N_i)
nij_by_nj = wordstats.get_nij_by_nj(collection, N_j)
thetas = np.array(range(1, max(N_i.A[0]) + 1))/N
opt_thetas = wordstats.get_opt_thetas(N, m, d, N_i, N_j, B_i, thetas)

Evaluate term dispersion scores.

In [6]:
# Calculate word dispersion scores according the various measures used in this study
IDF = wordstats.get_IDF(DF)
ICF = wordstats.get_ICF(CF)
Chisq = wordstats.get_Chisq(collection)
CG = wordstats.get_CG(N_i, B_i)
ICB = wordstats.get_ICB(nij_by_nj, B_i)
DoP = wordstats.get_DoP(collection, N_i, N_j, N)
RICF = wordstats.get_RICF(opt_thetas, N, ICF)

/Users/pasheridan/Desktop/github-repos/bursty-term-measure/genia/2-tables/singletons-excluded-analysis/../../../wordstats.py:209: RuntimeWarning: divide by zero encountered in log
  return -np.log(chisq_values)


Arrange term dispersion scores into a data frame.

In [7]:
# Initialize term dispersion scores data frame (augmented with ni and bi values)
term_scores_aug_df = pd.DataFrame(data=
                    {'lex': counter.get_feature_names_out(),
                     'IDF': IDF.A[0],
                     'ICF': ICF.A[0],
                     'Chi-sq': Chisq,
                     'CG': CG.A[0],
                     'ICB': ICB.A[0],
                     'DoP': DoP.A[0],
                     'RICF': RICF.A[0],
                     'bi': B_i.A[0],
                     'ni': N_i.A[0]})

# Augment with low-level and high-level semantic classes
term_scores_aug_df = pd.merge(term_scores_aug_df, genia_lexical_units_and_sems, on='lex', how='left')

# Tidy up the data frame
new_order = ['lex', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'RICF'] # Define column ordering
term_scores_aug_df = term_scores_aug_df.reindex(columns=new_order)
term_scores_aug_df = term_scores_aug_df.rename(columns={'lex': 'term'}) # Rename 'lex' column to 'term'
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')] # There are a few duplicate rows for some unknown reason
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

# Print to console
print("Term dispersion scores:")
display(term_scores_aug_df)

Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.404383
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.692896
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.808753


Term dispersion scores:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,-0.000251


In [8]:
# Integrate KeyBERT scores

# Load KeyBERT results 
keybert_keywords_path = '../keybert-scores.tsv'

with open(keybert_keywords_path, "r") as c:
  keybert_scores_df = pd.read_csv(c, sep='\t')

keybert_scores_df['sem'] = keybert_scores_df['sem'].fillna(str())

# Add KeyBERT scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keybert_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyBERT'] = term_scores_aug_df['KeyBERT'].fillna(0)

# Check for duplicate terms
all_duplicates = keybert_scores_df[keybert_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keybert_scores_df = keybert_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyBERT
3132,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,0.0010
16199,minus_clone_lex,G#cell_line,0.0005
18608,octamer_motif_lex,G#DNA_domain_or_region,0.0035


Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
16654,basic_helix-loop-helix_protein_lex,G#protein_family_or_group,amino_acid,4,6,6.214608,11.009194,310.072539,1.50,233.5,-0.001611,0.0010,0.404383
29778,minus_clone_lex,G#cell_line,cell,1,2,7.600902,12.107806,311.074036,2.00,466.0,-0.000643,0.0005,0.692896
32037,octamer_motif_lex,G#DNA_domain_or_region,nucleotide,8,18,5.521461,9.910582,inf,2.25,449.5,-0.004192,0.0035,0.808753


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40802,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,-0.000251
40803,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,-0.000251
40804,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,-0.000251
40805,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,-0.000251


In [9]:
# Integrate KeyLLM scores

# Load KeyLLM results 
keyllm_keywords_path = '../keyllm-scores.tsv'

with open(keyllm_keywords_path, "r") as c:
  keyllm_scores_df = pd.read_csv(c, sep='\t')

keyllm_scores_df['sem'] = keyllm_scores_df['sem'].fillna(str())

# Check for duplicate terms
all_duplicates = keyllm_scores_df[keyllm_scores_df.duplicated(subset=['term', 'sem'], keep='last')]
print("Duplicate rows:")
display(all_duplicates)
keyllm_scores_df = keyllm_scores_df.drop_duplicates(subset=['term', 'sem'], keep='last')

# Print to console
print("KeyLLM scores:")
display(keyllm_scores_df)

# Add KeyLLM scores to main results df
term_scores_aug_df = pd.merge(term_scores_aug_df, keyllm_scores_df, on=['term', 'sem'], how='left')
term_scores_aug_df['KeyLLM'] = term_scores_aug_df['KeyLLM'].fillna(0)

# Reorder columns
term_scores_aug_df = term_scores_aug_df.reindex(columns=['term', 'sem', 'class', 'bi', 'ni', 'IDF', 'ICF', 'Chi-sq', 'CG', 'ICB', 'DoP', 'KeyBERT', 'KeyLLM', 'RICF'])

# Check for duplicate terms
all_duplicates = term_scores_aug_df[term_scores_aug_df.duplicated(keep='first')]
print("Duplicate rows:")
display(all_duplicates)
term_scores_aug_df = term_scores_aug_df.drop_duplicates() # Drop any duplicate rows

display(term_scores_aug_df)

Duplicate rows:


,term,sem,KeyLLM
136,20-epi_analogue_lex,G#other_organic_compound,0.0005
137,20-Epi_analogue_lex,G#other_organic_compound,0.0005
177,25-dihydroxyvitamin_d3,,0.0015
192,25_dihydroxyvitamin_d3,,0.0005
413,9-cis_RA_lex,G#other_organic_compound,0.0005
...,...,...,...
18825,pRb_lex,G#protein_molecule,0.0010
18826,pRB_lex,G#protein_family_or_group,0.0010
22321,v-abl_lex,G#DNA_domain_or_region,0.0005
22322,v-Abl_lex,G#protein_molecule,0.0005


KeyLLM scores:


,term,sem,KeyLLM
0,(3H)_dexamethasone_lex,G#lipid,0.0005
1,(Ca2+)i_lex,G#inorganic,0.0005
2,(Ca2+)i_requirement_for_lex,G#other_name,0.0005
3,-120_region_lex,G#DNA_domain_or_region,0.0005
4,-130_AP-1-like_site_lex,G#DNA_domain_or_region,0.0005
...,...,...,...
33986,CD4_negative_T_cell_line_lex,G#cell_line,0.0000
33987,gp_160-induced_nuclear_extract_lex,G#cell_component,0.0000
33988,gp160-induced_AP-1_complex_lex,G#protein_complex,0.0000
33989,protein_synthesis-independent_lex,G#other_name,0.0000


Duplicate rows:


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF


,term,sem,class,bi,ni,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,'aged'_lymphocyte_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,244.0,-0.000673,0.0000,0.0,-0.000251
1,'converted'_TCEd_motif_lex,G#DNA_domain_or_region,nucleotide,1,1,7.600902,12.800954,0.701595,1.0,195.0,-0.000538,0.0000,0.0,-0.000251
2,'latency_I'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
3,'latency_II'_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,206.0,-0.000568,0.0000,0.0,-0.000251
4,'master_regulator_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,63.0,-0.000174,0.0000,0.0,-0.000251
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40799,zymogen_plasma_factor_X_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40800,zymogen_plasma_factors_VII_lex,G#protein_family_or_group,amino_acid,1,1,7.600902,12.800954,0.701595,1.0,355.0,-0.000979,0.0005,0.0,-0.000251
40801,zymography_lex,G#other_name,other,1,1,7.600902,12.800954,0.701595,1.0,191.0,-0.000527,0.0000,0.0,-0.000251
40802,zymosan-treated_cell_lex,G#cell_type,cell,1,1,7.600902,12.800954,0.701595,1.0,225.0,-0.000621,0.0005,0.0,-0.000251


In [10]:
# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores.tsv', sep='\t', index=False)

## Compile GENIA Corpus Summary Statistics

This is the result of Table 3 from the manuscript.

In [12]:
# Desginated ordering for the high-level semantic classes
high_level_semantic_class_ord = ['amino_acid', 'nucleotide', 'multi_cell', 'cell', 'other']

# Count number of semantic subclasses in each high-level class
subclass_counts = [len(amino_acid_sems), len(nucleotide_sems), len(multi_cell_sems), len(cell_sems), len(other_sems)]

# Count number of distinct lexical units in each high-level semantic class
lex_unit_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['term'].nunique().reindex(high_level_semantic_class_ord).to_list()

# Count number of annotations associated with each high-level semantic class
annotation_counts = term_scores_aug_df.dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Count number of singletons associated with each high-level semantic class
singleton_counts = term_scores_aug_df[term_scores_aug_df['ni'] == 1].dropna(subset=['class']).groupby('class')['ni'].sum().reindex(high_level_semantic_class_ord).to_list()

# Initialize GENIA summary statistics data frame
genia_summary_stats_df = pd.DataFrame({
    'Semantic class': high_level_semantic_class_ord,
    'Sub-class': subclass_counts,
    'Unique terms': lex_unit_counts,
    'Annotations': annotation_counts,
    'Singletons': singleton_counts
})

# Print GENIA summary statistics to console
display(genia_summary_stats_df)

,Semantic class,Sub-class,Unique terms,Annotations,Singletons
0,amino_acid,15,10155,42478,6571
1,nucleotide,12,5574,11619,4115
2,multi_cell,5,1444,5247,961
3,cell,4,4051,11626,2956
4,other,1,10560,19999,8071


## Terminology Extraction Task Experiment

Here we reproduce the results tables for the GENIA analysis.

In [13]:
# Create a minimal data frame of term dispersion scores
term_scores_df = term_scores_aug_df[term_scores_aug_df['ni'] > 1] # Filter out singletons
term_scores_df = term_scores_df.reset_index(drop=True) # Reinitialize row indices
term_scores_df = term_scores_df.drop(columns=['sem', 'class', 'ni', 'bi'])

# Print to console
display(term_scores_df)

# Write scores data frame to TSV
term_scores_aug_df.to_csv('term-dispersion-scores-minimal.tsv', sep='\t', index=False)

,term,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,(+)-pentazocine_lex,7.600902,12.107806,311.074036,2.0,522.0,-0.000720,0.0000,0.0000,0.692896
1,-120_lex,7.600902,12.107806,311.074036,2.0,360.0,-0.000496,0.0000,0.0000,0.692896
2,-150_bp_lex,7.600902,11.702341,inf,3.0,711.0,-0.000654,0.0000,0.0000,1.098361
3,-201/-184_NXS_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.0000,0.0005,0.692896
4,-201_and_-130_lex,7.600902,12.107806,311.074036,2.0,482.0,-0.000665,0.0000,0.0000,0.692896
...,...,...,...,...,...,...,...,...,...,...
14149,zinc_finger_region_lex,6.907755,12.107806,0.688948,1.0,200.5,-0.001106,0.0010,0.0005,-0.000532
14150,zinc_finger_transcription_factor_lex,5.991465,10.855043,122.211280,1.4,201.4,-0.002168,0.0025,0.0025,0.335116
14151,zinc_lex,6.907755,10.721512,inf,4.0,447.0,-0.000576,0.0010,0.0010,1.385763
14152,zone,6.907755,12.107806,0.688948,1.0,334.5,-0.001845,0.0000,0.0000,-0.000532


Define various functions used in the analysis.

In [14]:
# Grab the top k terms
def top_k(dct, k):
  keys = dct.keys()
  values = []
  for key in keys:
    values.append(dct[key][:k])
  keys_values_pair = zip(keys, values)
  return dict(keys_values_pair)

# Count up terms
def count_words(lst, imp_words):
  counter = 0
  for x in lst:
    if x in imp_words:
      counter += 1
  return counter

# Randomly resort term dispersion scores data frame
def resort(term_scores_df):
  sorted_terms = []
  bursty_measure_names = term_scores_df.columns.values.tolist()[1:]

  for measure in bursty_measure_names:
      # Copy the data frame and add a random column
      temp_df = term_scores_df.copy()
      temp_df['random'] = np.random.rand(len(temp_df))
        
      # Sort by the measure and the random column
      sorted_df = temp_df[['term', measure, 'random']].sort_values(by=[measure, 'random'], ascending=[False, True])
        
      # Append the sorted terms to the list
      sorted_terms.append(np.array(sorted_df['term']))
        
      # Drop the random column from the temporary data frame
      temp_df.drop(columns='random', inplace=True)
    
  sorted_terms = np.array(sorted_terms)
  measure_term_pair = zip(bursty_measure_names, sorted_terms)
  sorted_measures = dict(measure_term_pair)
    
  return sorted_measures
    
# Calculate Precision at k scores
def calc_pk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  pk_dct = dict(zip(measures, counts))  
  for measure in pk_dct.keys():
      for k in k_values:
          pk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/k)
  result = pd.DataFrame(pk_dct, index=k_values)
  return result

# Calculate Recall at k scores
def calc_rk(lex_units, sorted_measures, k_values):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rk_dct = dict(zip(measures, counts))  
  for measure in rk_dct.keys():
      for k in k_values:
          rk_dct[measure].append(count_words(top_k(sorted_measures, k)[measure], lex_units)/len(lex_units))
  result = pd.DataFrame(rk_dct, index=k_values)
  return result

# Calculate F1 at k scores
def calc_fk(pk, rk):
  result = 2 * (pk * rk) / (pk + rk)
  result = result.fillna(0) # Nan scores are redefined as 0
  return result

# Calculate Rank Biased Overlap scores
def calc_rbo(sorted_measures, k_values):
  RICF = sorted_measures["RICF"]
  measures = sorted_measures.keys()
  #print(measures)
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))  
  for measure in rbo_dct.keys():
      for k in k_values:
          S = top_k(sorted_measures, k)[measure]
          T = RICF[0:k]
          rbo_dct[measure].append(rbo.RankingSimilarity(S, T).rbo())
  result = pd.DataFrame(rbo_dct, index=k_values)
  return result

# Calculate Rank Biased Overlap scores for each semantic class
def calc_rbo2(sorted_measures, categories):
  measures = sorted_measures.keys()
  counts = [[], [], [], [], [], [], [], [], [], []]
  rbo_dct = dict(zip(measures, counts))

  for measure in measures:
      l1 = sorted_measures[measure].tolist()
      
      for category, lex_units in categories.items():
          S = sorted(set(l1) & set(lex_units), key = l1.index)
          l2 = sorted_measures["RICF"].tolist()
          RICF = sorted(set(l2) & set(lex_units), key = l2.index)          
          rbo_dct[measure].append(rbo.RankingSimilarity(S, RICF).rbo()) 

  result = pd.DataFrame(rbo_dct, index=categories.keys())
  return result
    
# Calculate mean P@k, R@k, and F1@k scores
def calc_score_means(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        means = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            mean_values = np.mean(values, axis=0)
            means[column] = mean_values
        result.append(pd.DataFrame(means, index=nested_list[0][i].index))
    return result

# Calculate standard deviations of P@k, R@k, and F1@k scores
def calc_score_sds(nested_list):
    result = []
    num_outer = len(nested_list)
    num_inner = len(nested_list[0])

    for i in range(num_inner):
        std_devs = {}
        for column in nested_list[0][i].columns:
            values = [nested_list[outer][i][column].values for outer in range(num_outer)]
            std_values = np.std(values, axis=0, ddof=1)
            std_devs[column] = std_values
        result.append(pd.DataFrame(std_devs, index=nested_list[0][i].index))
    return result

# Calculate mean RBO scores
def calc_mean_rbo_scores(scores_list):
  R = len(scores_list) # Number of replicates
  H = len(scores_list[0]) # Number of dispersion metrics
  d_metrics = scores_list[0].columns # Dispersion metrics by name
  k_values = scores_list[0].index # Top k values
  result = {}
    
  for d_metric in d_metrics:
    scores = [scores_list[r][d_metric].values for r in range(R)]
    mean_scores = np.mean(scores, axis=0)
    result[d_metric] = mean_scores
        
  return pd.DataFrame(result, index=k_values)

Evaluate Precision at k, Recall at k, F1 at k, and RBO scores.

In [15]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Set number of replicates
R = 100 # To test, set to 5

# These are the Precision @ k, Recall @ k and Rank Biased Overlap scores
all_pk_scores = []
all_rk_scores = []
all_fk_scores = []
all_rbo_scores = []
all_rbo_scores2 = []

# Used as inputs for calculating the various scores
k_values = np.array([10, 50, 100, 500, 1000, 5000])
amino_acid = get_high_level_semantic_class_words(amino_acid_sems)
nucleotide = get_high_level_semantic_class_words(nucleotide_sems)
multi_cell = get_high_level_semantic_class_words(multi_cell_sems)
cell = get_high_level_semantic_class_words(cell_sems)
other = get_high_level_semantic_class_words(other_sems)
categories = {
    'all': genia_lexical_units,
    'amino_acid': amino_acid,
    'nucleotide': nucleotide,
    'multi_cell': multi_cell,
    'cell': cell,
    'other': other}

# Calculate evaluation metrics
for r in tqdm(range(R)):
    print('r =', r)
    pk_scores = []
    rk_scores = []
    fk_scores = []
    sorted_measures = resort(term_scores_df)
    
    for category, lex_units in categories.items():
        pk = calc_pk(lex_units, sorted_measures, k_values)
        pk_scores.append(pk)
        rk = calc_rk(lex_units, sorted_measures, k_values)
        rk_scores.append(rk)
        fk = calc_fk(pk, rk)
        fk_scores.append(fk)
    
    all_pk_scores.append(pk_scores)
    all_rk_scores.append(rk_scores)
    all_fk_scores.append(fk_scores)
    rbo_scores = calc_rbo(sorted_measures, k_values)
    all_rbo_scores.append(rbo_scores)
    rbo_scores2 = calc_rbo2(sorted_measures, categories)
    all_rbo_scores2.append(rbo_scores2)

  0%|                                                                                                                                                | 0/100 [00:00<?, ?it/s]

r = 0


  1%|█▎                                                                                                                                    | 1/100 [00:57<1:35:07, 57.65s/it]

r = 1


  2%|██▋                                                                                                                                   | 2/100 [01:55<1:34:09, 57.65s/it]

...

 98%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎  | 98/100 [1:32:52<01:54, 57.05s/it]

r = 98


 99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋ | 99/100 [1:33:49<00:57, 57.19s/it]

r = 99


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:34:46<00:00, 56.87s/it]


Save evaluation metrics as Pkl files.

In [16]:
# Ensure the directory exists
os.makedirs('scores-dump', exist_ok=True)

# Write P@k scores to Pkl
with open('scores-dump/all_pk_scores.pkl', 'wb') as file:
    pickle.dump(all_pk_scores, file)

# Write R@k scores to Pkl
with open('scores-dump/all_rk_scores.pkl', 'wb') as file:
    pickle.dump(all_rk_scores, file)

# Write F1@k scores to Pkl
with open('scores-dump/all_fk_scores.pkl', 'wb') as file:
    pickle.dump(all_fk_scores, file)

# Write RBO scores to Pkl
with open('scores-dump/all_rbo_scores.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores, file)

# Write RBO scores as calculated for each semantic class to Pkl
with open('scores-dump/all_rbo_scores2.pkl', 'wb') as file:
    pickle.dump(all_rbo_scores2, file)

In [17]:
# Calculate mean P@k scores and write to CSV
all_pk_scores_means = calc_score_means(all_pk_scores)
os.makedirs('table-6', exist_ok=True)
pd.DataFrame(all_pk_scores_means[0]).to_csv('table-6/all-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[1]).to_csv('table-6/amino_acid-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[2]).to_csv('table-6/nucleotide-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[3]).to_csv('table-6/multicell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[4]).to_csv('table-6/cell-pk-means.csv', index=False)
pd.DataFrame(all_pk_scores_means[5]).to_csv('table-6/other-pk-means.csv', index=False)

# Calculate standard deviations for P@k scores and write to CSV
all_pk_scores_sds = calc_score_sds(all_pk_scores)
os.makedirs('table-a1', exist_ok=True)
pd.DataFrame(all_pk_scores_sds[0]).to_csv('table-a1/all-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[1]).to_csv('table-a1/amino_acid-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[2]).to_csv('table-a1/nucleotide-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[3]).to_csv('table-a1/multicell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[4]).to_csv('table-a1/cell-pk-sds.csv', index=False)
pd.DataFrame(all_pk_scores_sds[5]).to_csv('table-a1/other-pk-sds.csv', index=False)

In [18]:
# Display mean P@k scores and console
print("Mean P@k scores:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_means[0].round(4))
    display(all_pk_scores_means[1].round(4))
    display(all_pk_scores_means[2].round(4))
    display(all_pk_scores_means[3].round(4))
    display(all_pk_scores_means[4].round(4))
    display(all_pk_scores_means[5].round(4))

Mean P@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.9310,0.7340,0.9530,1.0000,1.0000,1.0000,1.0,1.0,1.0000
50,0.9280,0.7578,0.9508,0.9600,1.0000,0.9600,1.0,1.0,1.0000
100,0.9275,0.7651,0.9489,0.9800,0.9800,0.9400,1.0,1.0,1.0000
500,0.9266,0.7669,0.9534,0.9835,0.9740,0.9560,1.0,1.0,0.9914
1000,0.9275,0.7678,0.9533,0.9812,0.9633,0.9513,1.0,1.0,0.9853
5000,0.8800,0.7683,0.9151,0.9280,0.8992,0.8882,1.0,1.0,0.9314


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.4000,0.2390,0.5280,1.0000,0.8000,0.2000,0.5000,0.5000,1.0000
50,0.3882,0.2480,0.5398,0.7538,0.7000,0.4000,0.4000,0.5400,0.7906
100,0.3895,0.2493,0.5410,0.8200,0.7453,0.4600,0.4000,0.5071,0.8300
500,0.3882,0.2495,0.5400,0.6893,0.6220,0.4334,0.4578,0.4696,0.6926
1000,0.3909,0.2502,0.5382,0.6447,0.5899,0.4182,0.4437,0.4507,0.6436
5000,0.3567,0.2514,0.4293,0.4286,0.4144,0.3614,0.3736,0.4047,0.4294


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1720,0.1300,0.1520,0.0000,0.1000,0.2000,0.1000,0.0000,0.0000
50,0.1666,0.1388,0.1552,0.1000,0.1200,0.1200,0.0800,0.0400,0.1000
100,0.1688,0.1365,0.1472,0.1000,0.1000,0.1300,0.0700,0.0966,0.1100
500,0.1693,0.1323,0.1511,0.1408,0.1460,0.1346,0.0907,0.1186,0.1425
1000,0.1681,0.1320,0.1528,0.1344,0.1395,0.1580,0.1131,0.1253,0.1367
5000,0.1556,0.1327,0.1526,0.1551,0.1514,0.1552,0.1442,0.1472,0.1557


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0450,0.0360,0.0350,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0480,0.0388,0.0366,0.0400,0.0400,0.0200,0.1200,0.0600,0.0400
100,0.0490,0.0396,0.0383,0.0200,0.0200,0.0200,0.0800,0.0500,0.0200
500,0.0447,0.0423,0.0391,0.0330,0.0320,0.0660,0.0503,0.0355,0.0334
1000,0.0442,0.0420,0.0401,0.0369,0.0419,0.0580,0.0540,0.0384,0.0354
5000,0.0435,0.0410,0.0438,0.0438,0.0410,0.0454,0.0516,0.0496,0.0441


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1040,0.0910,0.0880,0.0000,0.000,0.1000,0.2000,0.2000,0.0000
50,0.1052,0.0936,0.0790,0.0062,0.000,0.0800,0.1600,0.1800,0.0094
100,0.1018,0.0962,0.0790,0.0100,0.020,0.0600,0.1700,0.1500,0.0100
500,0.1060,0.0961,0.0816,0.0389,0.060,0.0880,0.1591,0.1591,0.0400
1000,0.1056,0.0966,0.0823,0.0595,0.069,0.0910,0.1434,0.1447,0.0598
5000,0.1012,0.0949,0.0955,0.1000,0.101,0.0994,0.1247,0.1211,0.1007


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.2100,0.2380,0.1500,0.0000,0.1000,0.5000,0.2000,0.3000,0.0000
50,0.2200,0.2386,0.1402,0.0600,0.1400,0.3400,0.2400,0.1800,0.0600
100,0.2184,0.2435,0.1434,0.0300,0.0947,0.2700,0.2800,0.1963,0.0300
500,0.2184,0.2468,0.1416,0.0815,0.1140,0.2340,0.2420,0.2172,0.0830
1000,0.2188,0.2471,0.1400,0.1057,0.1229,0.2262,0.2458,0.2409,0.1098
5000,0.2230,0.2483,0.1938,0.2005,0.1913,0.2268,0.3059,0.2774,0.2015


In [19]:
# Displaye standard deviation of P@k scores to console
print("P@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_pk_scores_sds[0].round(4))
    display(all_pk_scores_sds[1].round(4))
    display(all_pk_scores_sds[2].round(4))
    display(all_pk_scores_sds[3].round(4))
    display(all_pk_scores_sds[4].round(4))
    display(all_pk_scores_sds[5].round(4))

P@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0825,0.1430,0.0611,0.0000,0.0000,0.0000,0.0,0.0,0.0000
50,0.0348,0.0634,0.0273,0.0000,0.0000,0.0000,0.0,0.0,0.0000
100,0.0230,0.0461,0.0219,0.0000,0.0000,0.0000,0.0,0.0,0.0000
500,0.0103,0.0181,0.0089,0.0009,0.0000,0.0000,0.0,0.0,0.0010
1000,0.0060,0.0110,0.0058,0.0015,0.0011,0.0005,0.0,0.0,0.0012
5000,0.0026,0.0004,0.0007,0.0007,0.0000,0.0000,0.0,0.0,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1633,0.1421,0.1511,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0700,0.0561,0.0719,0.0093,0.0000,0.0000,0.0000,0.0000,0.0100
100,0.0511,0.0389,0.0535,0.0000,0.0050,0.0000,0.0000,0.0046,0.0000
500,0.0210,0.0182,0.0217,0.0034,0.0000,0.0009,0.0031,0.0024,0.0042
1000,0.0154,0.0120,0.0140,0.0052,0.0007,0.0006,0.0033,0.0035,0.0039
5000,0.0022,0.0004,0.0010,0.0011,0.0001,0.0000,0.0023,0.0027,0.0005


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1349,0.1068,0.1059,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0568,0.0489,0.0477,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0357,0.0367,0.0359,0.0000,0.0000,0.0000,0.0000,0.0048,0.0000
500,0.0163,0.0163,0.0159,0.0021,0.0000,0.0009,0.0020,0.0015,0.0024
1000,0.0094,0.0098,0.0102,0.0034,0.0005,0.0000,0.0020,0.0023,0.0024
5000,0.0019,0.0003,0.0008,0.0010,0.0001,0.0000,0.0017,0.0022,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0672,0.0595,0.0539,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0299,0.0254,0.0297,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
100,0.0190,0.0174,0.0210,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0069,0.0093,0.0079,0.0014,0.0000,0.0,0.0015,0.0009,0.0014
1000,0.0051,0.0061,0.0047,0.0018,0.0007,0.0,0.0017,0.0014,0.0014
5000,0.0012,0.0002,0.0004,0.0005,0.0000,0.0,0.0011,0.0014,0.0002


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1024,0.0830,0.0946,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0468,0.0388,0.0417,0.0093,0.0000,0.0,0.0000,0.0000,0.0100
100,0.0320,0.0289,0.0277,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0133,0.0127,0.0099,0.0021,0.0000,0.0,0.0025,0.0012,0.0023
1000,0.0091,0.0087,0.0073,0.0025,0.0006,0.0,0.0024,0.0025,0.0019
5000,0.0015,0.0003,0.0007,0.0009,0.0001,0.0,0.0017,0.0018,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.1382,0.1405,0.1124,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0582,0.0574,0.0524,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0401,0.0402,0.0342,0.0000,0.0050,0.0000,0.0000,0.0049,0.0000
500,0.0191,0.0204,0.0136,0.0021,0.0000,0.0000,0.0032,0.0020,0.0028
1000,0.0109,0.0125,0.0092,0.0035,0.0007,0.0006,0.0030,0.0027,0.0025
5000,0.0023,0.0004,0.0009,0.0013,0.0001,0.0000,0.0022,0.0024,0.0006


In [20]:
# Calculate mean R@k scores and write to CSV
all_rk_scores_means = calc_score_means(all_rk_scores)
os.makedirs('table-7', exist_ok=True)
pd.DataFrame(all_rk_scores_means[0]).to_csv('table-7/all-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[1]).to_csv('table-7/amino_acid-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[2]).to_csv('table-7/nucleotide-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[3]).to_csv('table-7/multicell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[4]).to_csv('table-7/cell-rk-means.csv', index=False)
pd.DataFrame(all_rk_scores_means[5]).to_csv('table-7/other-rk-means.csv', index=False)

# Calculate standard deviations for R@k scores and write to CSV
all_rk_scores_sds = calc_score_sds(all_rk_scores)
os.makedirs('table-a2', exist_ok=True)
pd.DataFrame(all_rk_scores_sds[0]).to_csv('table-a2/all-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[1]).to_csv('table-a2/amino_acid-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[2]).to_csv('table-a2/nucleotide-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[3]).to_csv('table-a2/multicell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[4]).to_csv('table-a2/cell-rk-sds.csv', index=False)
pd.DataFrame(all_rk_scores_sds[5]).to_csv('table-a2/other-rk-sds.csv', index=False)

In [21]:
# Display mean R@k scores console
print("Mean R@k scores:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_means[0].round(4))
    display(all_rk_scores_means[1].round(4))
    display(all_rk_scores_means[2].round(4))
    display(all_rk_scores_means[3].round(4))
    display(all_rk_scores_means[4].round(4))
    display(all_rk_scores_means[5].round(4))

Mean R@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0002,0.0003,0.0003,0.0003,0.0003,0.0003,0.0003,0.0003
50,0.0015,0.0012,0.0015,0.0015,0.0016,0.0015,0.0016,0.0016,0.0016
100,0.0029,0.0024,0.0030,0.0031,0.0031,0.0030,0.0031,0.0031,0.0031
500,0.0146,0.0121,0.0150,0.0155,0.0153,0.0150,0.0157,0.0157,0.0156
1000,0.0292,0.0242,0.0300,0.0309,0.0303,0.0299,0.0315,0.0315,0.0310
5000,0.1384,0.1208,0.1439,0.1460,0.1414,0.1397,0.1573,0.1573,0.1465


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0004,0.0002,0.0005,0.0010,0.0008,0.0002,0.0005,0.0005,0.0010
50,0.0019,0.0012,0.0027,0.0037,0.0034,0.0020,0.0020,0.0027,0.0039
100,0.0038,0.0025,0.0053,0.0081,0.0073,0.0045,0.0039,0.0050,0.0082
500,0.0191,0.0123,0.0266,0.0339,0.0306,0.0213,0.0225,0.0231,0.0341
1000,0.0385,0.0246,0.0530,0.0635,0.0581,0.0412,0.0437,0.0444,0.0634
5000,0.1756,0.1238,0.2114,0.2110,0.2041,0.1779,0.1839,0.1993,0.2114


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0002,0.0003,0.0000,0.0002,0.0004,0.0002,0.0000,0.0000
50,0.0015,0.0012,0.0014,0.0009,0.0011,0.0011,0.0007,0.0004,0.0009
100,0.0030,0.0024,0.0026,0.0018,0.0018,0.0023,0.0013,0.0017,0.0020
500,0.0152,0.0119,0.0136,0.0126,0.0131,0.0121,0.0081,0.0106,0.0128
1000,0.0302,0.0237,0.0274,0.0241,0.0250,0.0283,0.0203,0.0225,0.0245
5000,0.1396,0.1190,0.1369,0.1391,0.1358,0.1392,0.1293,0.1320,0.1397


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0002,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0017,0.0013,0.0013,0.0014,0.0014,0.0007,0.0042,0.0021,0.0014
100,0.0034,0.0027,0.0027,0.0014,0.0014,0.0014,0.0055,0.0035,0.0014
500,0.0155,0.0146,0.0135,0.0114,0.0111,0.0229,0.0174,0.0123,0.0116
1000,0.0306,0.0291,0.0278,0.0255,0.0290,0.0402,0.0374,0.0266,0.0245
5000,0.1506,0.1421,0.1516,0.1518,0.1420,0.1572,0.1787,0.1716,0.1527


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0002,0.0002,0.0000,0.0000,0.0002,0.0005,0.0005,0.0000
50,0.0013,0.0012,0.0010,0.0001,0.0000,0.0010,0.0020,0.0022,0.0001
100,0.0025,0.0024,0.0020,0.0002,0.0005,0.0015,0.0042,0.0037,0.0002
500,0.0131,0.0119,0.0101,0.0048,0.0074,0.0109,0.0196,0.0196,0.0049
1000,0.0261,0.0238,0.0203,0.0147,0.0170,0.0225,0.0354,0.0357,0.0148
5000,0.1248,0.1171,0.1179,0.1234,0.1247,0.1227,0.1540,0.1494,0.1243


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0002,0.0002,0.0001,0.0000,0.0001,0.0005,0.0002,0.0003,0.0000
50,0.0010,0.0011,0.0007,0.0003,0.0007,0.0016,0.0011,0.0009,0.0003
100,0.0021,0.0023,0.0014,0.0003,0.0009,0.0026,0.0027,0.0019,0.0003
500,0.0103,0.0117,0.0067,0.0039,0.0054,0.0111,0.0115,0.0103,0.0039
1000,0.0207,0.0234,0.0133,0.0100,0.0116,0.0214,0.0233,0.0228,0.0104
5000,0.1056,0.1176,0.0918,0.0949,0.0906,0.1074,0.1448,0.1314,0.0954


In [22]:
# Display standard deviation of R@k scores to console
print("R@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_rk_scores_sds[0].round(4))
    display(all_rk_scores_sds[1].round(4))
    display(all_rk_scores_sds[2].round(4))
    display(all_rk_scores_sds[3].round(4))
    display(all_rk_scores_sds[4].round(4))
    display(all_rk_scores_sds[5].round(4))

R@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000
50,0.0001,0.0001,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000
100,0.0001,0.0001,0.0001,0.0000,0.0,0.0,0.0,0.0,0.0000
500,0.0002,0.0003,0.0001,0.0000,0.0,0.0,0.0,0.0,0.0000
1000,0.0002,0.0003,0.0002,0.0000,0.0,0.0,0.0,0.0,0.0000
5000,0.0004,0.0001,0.0001,0.0001,0.0,0.0,0.0,0.0,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0002,0.0001,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0003,0.0003,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0005,0.0004,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0010,0.0009,0.0011,0.0002,0.0000,0.0000,0.0002,0.0001,0.0002
1000,0.0015,0.0012,0.0014,0.0005,0.0001,0.0001,0.0003,0.0003,0.0004
5000,0.0011,0.0002,0.0005,0.0005,0.0000,0.0000,0.0012,0.0013,0.0002


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0002,0.0002,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0005,0.0004,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0006,0.0007,0.0006,0.0000,0.0000,0.0000,0.0000,0.0001,0.0000
500,0.0015,0.0015,0.0014,0.0002,0.0000,0.0001,0.0002,0.0001,0.0002
1000,0.0017,0.0018,0.0018,0.0006,0.0001,0.0000,0.0004,0.0004,0.0004
5000,0.0017,0.0002,0.0007,0.0009,0.0001,0.0000,0.0015,0.0020,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0004,0.0004,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0010,0.0009,0.0010,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
100,0.0013,0.0012,0.0015,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0024,0.0032,0.0027,0.0005,0.0000,0.0,0.0005,0.0003,0.0005
1000,0.0035,0.0042,0.0033,0.0013,0.0005,0.0,0.0012,0.0010,0.0010
5000,0.0041,0.0006,0.0015,0.0018,0.0002,0.0,0.0038,0.0048,0.0009


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0002,0.0002,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0006,0.0005,0.0005,0.0001,0.0000,0.0,0.0000,0.0000,0.0001
100,0.0008,0.0007,0.0007,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0016,0.0016,0.0012,0.0003,0.0000,0.0,0.0003,0.0002,0.0003
1000,0.0022,0.0021,0.0018,0.0006,0.0002,0.0,0.0006,0.0006,0.0005
5000,0.0019,0.0003,0.0009,0.0011,0.0001,0.0,0.0021,0.0023,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0001,0.0001,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0003,0.0003,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0004,0.0004,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
500,0.0009,0.0010,0.0006,0.0001,0.0000,0.0000,0.0001,0.0001,0.0001
1000,0.0010,0.0012,0.0009,0.0003,0.0001,0.0001,0.0003,0.0003,0.0002
5000,0.0011,0.0002,0.0004,0.0006,0.0000,0.0000,0.0010,0.0011,0.0003


In [23]:
# Calculate mean F1@k scores and write to CSV
all_fk_scores_means = calc_score_means(all_fk_scores)
os.makedirs('table-8', exist_ok=True)
pd.DataFrame(all_fk_scores_means[0]).to_csv('table-8/all-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[1]).to_csv('table-8/amino_acid-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[2]).to_csv('table-8/nucleotide-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[3]).to_csv('table-8/multicell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[4]).to_csv('table-8/cell-fk-means.csv', index=False)
pd.DataFrame(all_fk_scores_means[5]).to_csv('table-8/other-fk-means.csv', index=False)

# Calculate standard deviations for F1@k scores and write to CSV
all_fk_scores_sds = calc_score_sds(all_fk_scores)
os.makedirs('table-a3', exist_ok=True)
pd.DataFrame(all_fk_scores_sds[0]).to_csv('table-a3/all-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[1]).to_csv('table-a3/amino_acid-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[2]).to_csv('table-a3/nucleotide-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[3]).to_csv('table-a3/multicell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[4]).to_csv('table-a3/cell-fk-sds.csv', index=False)
pd.DataFrame(all_fk_scores_sds[5]).to_csv('table-a3/other-fk-sds.csv', index=False)

In [24]:
# Display mean F1@k scores to console
print("Mean F1@k scores:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_means[0].round(4))
    display(all_fk_scores_means[1].round(4))
    display(all_fk_scores_means[2].round(4))
    display(all_fk_scores_means[3].round(4))
    display(all_fk_scores_means[4].round(4))
    display(all_fk_scores_means[5].round(4))

Mean F1@k scores:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0005,0.0006,0.0006,0.0006,0.0006,0.0006,0.0006,0.0006
50,0.0029,0.0024,0.0030,0.0030,0.0031,0.0030,0.0031,0.0031,0.0031
100,0.0058,0.0048,0.0060,0.0061,0.0061,0.0059,0.0063,0.0063,0.0063
500,0.0287,0.0238,0.0295,0.0305,0.0302,0.0296,0.0310,0.0310,0.0307
1000,0.0566,0.0468,0.0582,0.0599,0.0588,0.0580,0.0610,0.0610,0.0601
5000,0.2392,0.2088,0.2488,0.2523,0.2444,0.2414,0.2718,0.2718,0.2532


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0008,0.0005,0.0010,0.0020,0.0016,0.0004,0.0010,0.0010,0.0020
50,0.0038,0.0024,0.0053,0.0074,0.0069,0.0039,0.0039,0.0053,0.0077
100,0.0076,0.0049,0.0106,0.0160,0.0145,0.0090,0.0078,0.0099,0.0162
500,0.0364,0.0234,0.0507,0.0647,0.0584,0.0407,0.0430,0.0441,0.0650
1000,0.0701,0.0449,0.0965,0.1156,0.1058,0.0750,0.0796,0.0808,0.1154
5000,0.2354,0.1659,0.2833,0.2828,0.2735,0.2385,0.2465,0.2670,0.2833


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0005,0.0005,0.0000,0.0004,0.0007,0.0004,0.0000,0.0000
50,0.0030,0.0025,0.0028,0.0018,0.0021,0.0021,0.0014,0.0007,0.0018
100,0.0059,0.0048,0.0052,0.0035,0.0035,0.0046,0.0025,0.0034,0.0039
500,0.0279,0.0218,0.0249,0.0232,0.0240,0.0222,0.0149,0.0195,0.0235
1000,0.0511,0.0401,0.0465,0.0409,0.0424,0.0481,0.0344,0.0381,0.0416
5000,0.1471,0.1255,0.1443,0.1467,0.1432,0.1468,0.1364,0.1392,0.1473


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0005,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0032,0.0026,0.0024,0.0027,0.0027,0.0013,0.0080,0.0040,0.0027
100,0.0063,0.0051,0.0050,0.0026,0.0026,0.0026,0.0104,0.0065,0.0026
500,0.0230,0.0217,0.0201,0.0170,0.0165,0.0340,0.0259,0.0183,0.0172
1000,0.0361,0.0343,0.0328,0.0302,0.0343,0.0475,0.0442,0.0314,0.0289
5000,0.0675,0.0637,0.0680,0.0680,0.0636,0.0705,0.0801,0.0769,0.0684


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0004,0.0004,0.0000,0.0000,0.0005,0.0010,0.0010,0.0000
50,0.0026,0.0023,0.0019,0.0002,0.0000,0.0020,0.0039,0.0044,0.0002
100,0.0049,0.0046,0.0038,0.0005,0.0010,0.0029,0.0082,0.0072,0.0005
500,0.0233,0.0211,0.0179,0.0086,0.0132,0.0193,0.0350,0.0350,0.0088
1000,0.0418,0.0382,0.0326,0.0236,0.0273,0.0360,0.0568,0.0573,0.0237
5000,0.1118,0.1048,0.1056,0.1105,0.1116,0.1098,0.1378,0.1338,0.1112


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0004,0.0005,0.0003,0.0000,0.0002,0.0009,0.0004,0.0006,0.0000
50,0.0021,0.0022,0.0013,0.0006,0.0013,0.0032,0.0023,0.0017,0.0006
100,0.0041,0.0046,0.0027,0.0006,0.0018,0.0051,0.0053,0.0037,0.0006
500,0.0198,0.0223,0.0128,0.0074,0.0103,0.0212,0.0219,0.0196,0.0075
1000,0.0378,0.0427,0.0242,0.0183,0.0213,0.0391,0.0425,0.0417,0.0190
5000,0.1433,0.1596,0.1246,0.1289,0.1229,0.1458,0.1966,0.1783,0.1295


In [25]:
# Display standard deviation of F1@k scores to console
print("F1@k scores standard deviations:")
with pd.option_context('display.precision', 4):
    display(all_fk_scores_sds[0].round(4))
    display(all_fk_scores_sds[1].round(4))
    display(all_fk_scores_sds[2].round(4))
    display(all_fk_scores_sds[3].round(4))
    display(all_fk_scores_sds[4].round(4))
    display(all_fk_scores_sds[5].round(4))

F1@k scores standard deviations:


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0001,0.0001,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0000
50,0.0001,0.0002,0.0001,0.0000,0.0000,0.0,0.0,0.0,0.0000
100,0.0001,0.0003,0.0001,0.0000,0.0000,0.0,0.0,0.0,0.0000
500,0.0003,0.0006,0.0003,0.0000,0.0000,0.0,0.0,0.0,0.0000
1000,0.0004,0.0007,0.0004,0.0001,0.0001,0.0,0.0,0.0,0.0001
5000,0.0007,0.0001,0.0002,0.0002,0.0000,0.0,0.0,0.0,0.0001


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0003,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0007,0.0006,0.0007,0.0001,0.0000,0.0000,0.0000,0.0000,0.0001
100,0.0010,0.0008,0.0010,0.0000,0.0001,0.0000,0.0000,0.0001,0.0000
500,0.0020,0.0017,0.0020,0.0003,0.0000,0.0001,0.0003,0.0002,0.0004
1000,0.0028,0.0021,0.0025,0.0009,0.0001,0.0001,0.0006,0.0006,0.0007
5000,0.0014,0.0002,0.0007,0.0007,0.0001,0.0000,0.0016,0.0018,0.0003


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0004,0.0004,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0010,0.0009,0.0008,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0013,0.0013,0.0013,0.0000,0.0000,0.0000,0.0000,0.0002,0.0000
500,0.0027,0.0027,0.0026,0.0003,0.0000,0.0002,0.0003,0.0002,0.0004
1000,0.0029,0.0030,0.0031,0.0010,0.0002,0.0000,0.0006,0.0007,0.0007
5000,0.0018,0.0003,0.0008,0.0009,0.0001,0.0000,0.0016,0.0021,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0009,0.0008,0.0007,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0020,0.0017,0.0020,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
100,0.0025,0.0023,0.0027,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0036,0.0048,0.0041,0.0007,0.0000,0.0,0.0008,0.0005,0.0007
1000,0.0041,0.0050,0.0038,0.0015,0.0006,0.0,0.0014,0.0011,0.0011
5000,0.0018,0.0003,0.0007,0.0008,0.0001,0.0,0.0017,0.0021,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0005,0.0004,0.0005,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
50,0.0011,0.0009,0.0010,0.0002,0.0000,0.0,0.0000,0.0000,0.0002
100,0.0015,0.0014,0.0013,0.0000,0.0000,0.0,0.0000,0.0000,0.0000
500,0.0029,0.0028,0.0022,0.0005,0.0000,0.0,0.0005,0.0003,0.0005
1000,0.0036,0.0034,0.0029,0.0010,0.0003,0.0,0.0009,0.0010,0.0008
5000,0.0017,0.0003,0.0008,0.0010,0.0001,0.0,0.0018,0.0020,0.0004


,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0003,0.0003,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
50,0.0005,0.0005,0.0005,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
100,0.0008,0.0008,0.0006,0.0000,0.0001,0.0000,0.0000,0.0001,0.0000
500,0.0017,0.0018,0.0012,0.0002,0.0000,0.0000,0.0003,0.0002,0.0003
1000,0.0019,0.0022,0.0016,0.0006,0.0001,0.0001,0.0005,0.0005,0.0004
5000,0.0015,0.0002,0.0006,0.0008,0.0001,0.0000,0.0014,0.0015,0.0004


In [26]:
# Calculate mean RBO scores
mean_rbo_scores = calc_mean_rbo_scores(all_rbo_scores)

# Write to CSV
os.makedirs('table-9', exist_ok=True)
pd.DataFrame(mean_rbo_scores).to_csv('table-9/rbo-means.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
10,0.0006,0.0000,0.0015,0.9140,0.6098,0.0000,0.0000,0.0000,1.0
50,0.0042,0.0000,0.0074,0.9001,0.6846,0.0000,0.0000,0.0000,1.0
100,0.0099,0.0000,0.0165,0.9093,0.7005,0.0000,0.0000,0.0000,1.0
500,0.0526,0.0000,0.0986,0.9313,0.7284,0.0148,0.0025,0.0012,1.0
1000,0.0988,0.0000,0.1988,0.9356,0.7655,0.0561,0.0169,0.0133,1.0
5000,0.4517,0.1289,0.6537,0.9064,0.8015,0.4215,0.1393,0.1383,1.0


In [27]:
# Calculate mean RBO scores by semantic class
mean_rbo_scores2 = calc_mean_rbo_scores(all_rbo_scores2)

# Write to CSV
os.makedirs('table-10', exist_ok=True)
pd.DataFrame(mean_rbo_scores2).to_csv('table-10/rbo-means-by-semantic-class.csv', index=False)

# Display RBO scores to console
with pd.option_context('display.precision', 4):
    display(mean_rbo_scores2.round(4))

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
all,0.6312,0.4292,0.8137,0.9384,0.8665,0.6154,0.4120,0.4094,1.0
amino_acid,0.5973,0.3856,0.7750,0.9564,0.8818,0.5846,0.4370,0.4230,1.0
nucleotide,0.6446,0.4327,0.8070,0.9268,0.8645,0.6231,0.4045,0.4016,1.0
multi_cell,0.6427,0.4535,0.8310,0.9321,0.8481,0.6202,0.4335,0.4121,1.0
cell,0.6582,0.4757,0.8316,0.9202,0.8495,0.6423,0.3832,0.3782,1.0
other,0.6795,0.4822,0.8440,0.9049,0.8303,0.6594,0.3862,0.3871,1.0


## Top 10 Ranked Terms Example

Here we reproduce the result of Table 10 from the manuscript.

In [28]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Retrieve top 10 ranked terms
top = 10
ranked_terms_df = resort(term_scores_df)
top_10_ranked_terms_df = pd.DataFrame(top_k(ranked_terms_df, top))

# Print to console
display(top_10_ranked_terms_df)

# Write to CSV
os.makedirs('table-11', exist_ok=True)
top_10_ranked_terms_df.to_csv('table-11/top-10-terms.csv', index=False)

,IDF,ICF,Chi-sq,CG,ICB,DoP,KeyBERT,KeyLLM,RICF
0,V14Rho_lex,marginally,TCL1_lex,Bcl-6_lex,Bcl-6_lex,D_variant_cell_lex,transcription_factor_lex,transcription_factor_lex,Bcl-6_lex
1,lymphoid_progenitor_cell_lex,dispersed,RBP-Jkappa_lex,v-erbA_lex,TCRzeta_lex,combined_pituitary_hormone_deficiency_lex,activation_lex,T_cell_lex,SMX_lex
2,achievable,campomelic_dysplasia_lex,CD28_lex,SMX_lex,ML-9_lex,Pit-1_gene_lex,cytokine_lex,NF-kappa_B_lex,v-erbA_lex
3,cytokinestimulated,IL-2_receptor_beta-chain_lex,RAI_lex,SHP1_lex,AITL_lex,EBNA-5_lex,transcription_lex,cytokine_lex,SHP1_lex
4,empirically,tyrosine_kinase_receptor_lex,chromosome_16_lex,ML-9_lex,SHP1_lex,palmar_fibromatosis_lex,T_cell_lex,NF-kappaB_lex,ML-9_lex
5,Bcl-x(L)_lex,pancreatic,LMP_promoter_lex,beta-casein_lex,beta-casein_lex,differentiation-inducing_activity_lex,gene_lex,AP-1_lex,beta-casein_lex
6,SP100_lex,UT-7_Epo_cell_lex,ZEB_lex,I_kappaB_lex,A-myb_lex,Toremifene_lex,NF-kappa_B_lex,transcription_lex,TCRzeta_lex
7,bipolar_disorder_lex,caspase_inhibitor_lex,respiratory_epithelium_lex,p95vav_lex,I_kappaB_lex,lymphocyte_death_lex,Il-2_lex,gene_expression_lex,p95vav_lex
8,PB1_lex,mouse_AML1/PEBP2_alpha_B_lex,ZAP-70_lex,TCRzeta_lex,SMX_lex,DEN_lex,IL-2_lex,T_lymphocyte_lex,I_kappaB_lex
9,RANTES_gene_lex,TATA_element_lex,SN50_lex,EBNA-2_lex,Rap1_protein_lex,rickets-like_disease_lex,monocyte_lex,apoptosis_lex,DM_lex


## Stopwords Exploratory Analysis

Here we reproduce the result of Table 12 from the manuscript.

In [29]:
def getrank(sorted_measures):
    unique_terms = set()
    for terms in sorted_measures.values():
        unique_terms.update(terms)
    unique_terms = sorted(unique_terms)
    
    # Create a data frame to hold the rankings
    ranking_df = pd.DataFrame(index=unique_terms, columns=sorted_measures.keys())
    
    # Fill the data frame with rankings
    for measure, terms in sorted_measures.items():
        for rank, term in enumerate(terms):
            ranking_df.at[term, measure] = rank + 1  # Rank starts from 1
    
    # Replace NaN with a large number to indicate unranked terms
    ranking_df = ranking_df.fillna(len(unique_terms) + 1)
    #csv_file_path = 'ranking_table.csv'
    #ranking_df.to_csv(csv_file_path)
    return ranking_df

# Function to filter stopwords from the ranking data frame
def filter_stopwords(ranking_df):
    stopwords_list = set(stopwords.words('english'))
    
    # Filter the data frame to include only stopwords
    stopwords_rank = ranking_df[ranking_df.index.isin(stopwords_list)]
    
    # Save the stopwords ranking data frame to a CSV file
    #csv_file_path = 'stopwords_ranking_table.csv'
    #stopwords_rank.to_csv(csv_file_path)
    
    return stopwords_rank

In [30]:
# Initialize a random seed to ensure results can be replicated
np.random.seed(641369)

# Generate term dispersion ranks for R different versions of the data
all_quantiles_df = []
for r in tqdm(range(R)):
    sorted_measures = resort(term_scores_df)
    rank = getrank(sorted_measures)
    stopwords_ranks_df = filter_stopwords(rank)
    bursty_measure_names = stopwords_ranks_df.head(0)
    quantiles = []
    for bursty_measure_name in bursty_measure_names:
        quantiles.append(stopwords_ranks_df[bursty_measure_name].quantile([0, 0.25, 0.5, 0.75, 1]))
    quantiles_df = pd.DataFrame(quantiles)
    all_quantiles_df.append(quantiles_df)

# Extract the column and index names from the first quantiles data frame
columns = all_quantiles_df[0].columns
index = all_quantiles_df[0].index

# Initialize empty data frames to store the mean and standard deviation values
mean_df = pd.DataFrame(index=index, columns=columns)
std_df = pd.DataFrame(index=index, columns=columns)

# Compute the mean and standard deviation of corresponding elements across all matrices
for col in columns:
    for idx in index:
        values = [matrix.at[idx, col] for matrix in all_quantiles_df]
        mean_df.at[idx, col] = np.mean(values)
        std_df.at[idx, col] = np.std(values)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [00:52<00:00,  1.90it/s]


In [31]:
# Print to console
print("Mean values:")
with pd.option_context('display.precision', 4):
    display(mean_df)
print("\nStandard deviations:")
with pd.option_context('display.precision', 4):
    display(std_df)

# Write to CSV
os.makedirs('table-12', exist_ok=True)
mean_df.to_csv('table-12/stopword-rank-means.csv')
std_df.to_csv('table-12/stopword-rank-sds.csv')

Mean values:


,0.00,0.25,0.50,0.75,1.00
IDF,2046.22,13361.635,13968.485,14122.25,14154.0
ICF,920.67,13270.24,13936.53,14123.25,14154.0
Chi-sq,833.32,7237.0,8053.0,8649.5,14153.0
CG,19.0,5602.5,7690.45,8524.4175,13888.27
ICB,70.0,5375.0,7542.5,8941.25,13910.44
DoP,3232.93,13354.25,13971.0,14123.0,14154.0
KeyBERT,7139.04,8917.535,10594.085,12348.365,14094.21
KeyLLM,7551.38,9154.0975,10804.8,12480.73,14085.42
RICF,2427.0,7800.61,8224.5,8775.87,14153.0



Standard deviations:


,0.00,0.25,0.50,0.75,1.00
IDF,1117.0618,3.287,0.3497,0.0,0.0
ICF,715.4105,2.5676,0.6396,0.0,0.0
Chi-sq,643.3162,0.0,0.0,0.0,0.0
CG,0.0,0.0,0.6764,2.194,233.1068
ICB,0.0,0.0,0.0,0.0,3.0994
DoP,4.0774,0.0,0.0,0.0,0.0
KeyBERT,60.1609,270.5655,308.0247,292.1473,51.2254
KeyLLM,67.2574,293.0649,342.9913,278.9519,70.3399
RICF,0.0,0.3747,0.0,0.1249,0.0
